# Aral pipeline on Colab

Runs layer 1 of the Aral Sea land-cover pipeline where there is disk and
bandwidth to spare, and sends back only what cannot be recreated.

**What stays here and dies with the session:** raw scenes, grid tiles.
They are reproducible from the manifest in a few minutes, so they are not
worth storing anywhere.

**What goes to Drive:** the manifest, the QA table, and the patches for
hand labelling. Tens of megabytes. That is the part your own work depends on.

Runtime > Change runtime type > GPU is **not** needed for this notebook;
every stage here is network- and disk-bound.

## 1. Dependencies

In [ ]:
!pip -q install rasterio pandas shapely
import rasterio, pandas
print('rasterio', rasterio.__version__, '| GDAL', rasterio.__gdal_version__)

## 2. The code

Run this cell again after every change you push: it refreshes the code in
place and forces a re-import. Skipping the refresh when the directory
already exists is what makes a fixed bug look unfixed.

Only the code is replaced. `data/` is untracked and survives, so a refresh
never costs you a download.

**Private repo:** Colab holds none of your credentials, so a private
repository fails exactly like a missing one (exit 128). Put a fine-grained
token with *Contents: read* into Colab Secrets (key icon, left sidebar) as
`GITHUB_TOKEN`. It is never printed and never written to `.git/config`.

**No repo?** Leave `REPO_URL` empty and upload `dist/aral_src.zip` to
`MyDrive/aral/`.

In [ ]:
REPO_URL = 'https://github.com/xdido2/aral-geo-map.git'          # e.g. 'https://github.com/<you>/aral-geo-map.git'

import os, sys, pathlib, subprocess
from google.colab import drive
drive.mount('/content/drive')

DRIVE = pathlib.Path('/content/drive/MyDrive/aral')
DRIVE.mkdir(parents=True, exist_ok=True)
ROOT = pathlib.Path('/content/aral-geo-map')


def _token():
    """GitHub token from Colab Secrets, or None. Never reaches the output."""
    try:
        from google.colab import userdata
        return userdata.get('GITHUB_TOKEN')
    except Exception:
        return None


def _run(args, tok):
    r = subprocess.run(args, capture_output=True, text=True)
    if r.returncode:
        msg = (r.stderr or r.stdout).strip()
        print('git failed:\n ', msg.replace(tok, '***') if tok else msg)
        if not tok:
            print('  (a private repo looks like a missing one here -')
            print('   add GITHUB_TOKEN to Colab Secrets if it is private)')
    return r.returncode == 0


def from_git(url):
    """Clone, or bring an existing clone up to date. Token stays off disk."""
    tok = _token()
    auth = url.replace('https://', f'https://{tok}@') if tok else url
    if (ROOT / '.git').exists():
        # fetch by explicit URL, so no credential is ever stored in the repo
        if not _run(['git', '-C', str(ROOT), 'fetch', '--quiet', auth, 'main'], tok):
            return None
        # reset touches tracked files only; data/ is untracked and untouched
        if not _run(['git', '-C', str(ROOT), 'reset', '--hard', '--quiet',
                     'FETCH_HEAD'], tok):
            return None
        return f'{url} (updated)'
    if not _run(['git', 'clone', '--quiet', auth, str(ROOT)], tok):
        return None
    _run(['git', '-C', str(ROOT), 'remote', 'set-url', 'origin', url], tok)
    return f'{url} (cloned)'


def from_zip():
    src_zip = DRIVE / 'aral_src.zip'
    if not src_zip.exists():
        raise SystemExit(f'Upload dist/aral_src.zip to {DRIVE}, or set REPO_URL.')
    ROOT.mkdir(parents=True, exist_ok=True)
    # -o overwrites the code and leaves everything else alone
    subprocess.run(['unzip', '-qo', str(src_zip), '-d', str(ROOT)], check=True)
    return f'{src_zip.name} (unpacked)'


source = (from_git(REPO_URL) if REPO_URL else None) or from_zip()

os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
# Updating the files is not enough: a module already imported stays in
# memory, so a fix on disk would look like no fix at all.
for name in [m for m in list(sys.modules) if m.startswith('src.aral')]:
    del sys.modules[name]
from src.aral import config as C, grid, manifest, fetch, prepare, patches, qa, indices

print('source :', source)
print('assets :', fetch.ASSETS)
print('season :', C.SEASON, '| cells:', len(C.PATHS) * len(C.ROWS))

## 3. The analysis grid

A sanity check before anything is downloaded: if these numbers are not what
you expect, the AOI or the CRS in `config.py` is wrong, and every later
stage would inherit that.

In [ ]:
for k, v in grid.describe().items():
    print(f'{k:14s} {v}')

## 4. Manifest

Built once and kept on Drive, because it is the artefact that defines the
dataset: change the season or the cells and you have a different project.

The cell refuses to reuse a manifest whose recorded build parameters differ
from the config in force now. A stale one does not announce itself - it just
lacks scenes the code expects, and the failure surfaces stages later as a
missing file.

In [ ]:
import shutil, pandas as pd

C.INTERIM.mkdir(parents=True, exist_ok=True)
local = C.INTERIM / 'manifest.csv'

for name, dst in ((('manifest.csv'), local),
                  ('manifest.json', manifest.params_path(local))):
    if (DRIVE / name).exists():
        shutil.copy(DRIVE / name, dst)

if manifest.matches_config(local):
    print('manifest from Drive matches the current config')
else:
    print('no manifest, or built with other parameters - rebuilding')
    print(' want:', dict(season=C.SEASON, years=C.YEARS,
                         paths=C.PATHS, rows=C.ROWS))
    print(' have:', manifest.params(local))
    manifest.build()                       # ~3 min of catalogue queries
    shutil.copy(local, DRIVE / 'manifest.csv')
    shutil.copy(manifest.params_path(local), DRIVE / 'manifest.json')

df = pd.read_csv(local, dtype={'path': str, 'row': str})
print(len(df), 'scenes;', df.year.min(), '-', df.year.max())
df.head()

## 5. Choose a subset

Never fetch the whole manifest here: a Colab disk is ~100 GB and the session
ends without warning. One season of a few cells is what this notebook is for.

In [ ]:
YEAR = 2019
CELLS = [('161', '028'), ('161', '029')]      # the two that hold most water

sel = df[(df.year == YEAR) & (df.tier.isin(C.TIERS)) & (df.cloud <= C.MAX_CLOUD)]
sel = sel[[(p, r) in CELLS for p, r in zip(sel.path, sel['row'])]]
rows = sel.to_dict('records')
print(f'{len(rows)} scenes, {len(rows)*len(fetch.ASSETS)} assets, ~{len(rows)*0.3:.1f} GB whole / less when cropped')
sel[['product_id', 'date', 'path', 'row', 'cloud', 'sun_elev']]

## 6. Fetch

`window` crops each scene to the AOI as it reads, using range requests.
Re-running skips whatever is already on disk, so an interrupted cell is
resumed, not restarted.

In [ ]:
stats = fetch.fetch(rows, mode='window', workers=16)
print(stats)

## 7. Prepare

Validity decided on raw DN, nearest-neighbour onto the fixed grid, then
reflectance. The order is what keeps fill zeros out of the arithmetic.

In [ ]:
tiles, qa_rows = [], []
for r in rows:
    d = fetch.scene_dir(C.RAW / 'landsat', r)
    out = C.INTERIM / 'tiles' / f"{r['product_id']}.tif"
    if out.exists():
        st = {'product_id': r['product_id'], 'valid_frac': None}
    else:
        st = prepare.prepare(d, r['product_id'], out)
        if st is None:
            continue
    tiles.append(out)
    qa_rows.append(qa.tile_stats(out, {**st, 'path': r['path'], 'row': r['row'],
                                       'date': r['date']}))
    print(out.name, st.get('valid_frac'))
print(len(tiles), 'tiles')

## 8. Quality control

One row per tile. Sort it and the odd one out shows itself: a tile that is
mostly mask, or whose water fraction sits far from its neighbours in the
same cell on nearby dates.

In [ ]:
import pandas as pd
qdf = pd.DataFrame(qa_rows)
qa.write(qa_rows, C.REPORTS / 'tile_stats.csv')
qdf.to_csv(DRIVE / 'tile_stats.csv', index=False)
qdf.sort_values('water_frac')

### Look at one, before trusting any of them

In [ ]:
import numpy as np, matplotlib.pyplot as plt

with rasterio.open(tiles[0]) as s:
    step = max(s.height // 700, 1)
    a = s.read(masked=True, out_shape=(s.count, s.height // step, s.width // step))
rgb = np.ma.stack([a[4], a[3], a[2]]).astype('f4') / C.STORE_SCALE   # SWIR1/NIR/red
rgb = np.clip((rgb - 0.0) / 0.45, 0, 1).transpose(1, 2, 0)
plt.figure(figsize=(7, 7)); plt.imshow(rgb); plt.axis('off')
plt.title(tiles[0].stem[:40]); plt.show()

## 9. Patches for labelling

This is the output that matters. Patches are named by their position on the
global grid, so `x012y035` is the same ground in every year, and the set can
be extended later without renaming anything.

About 200 KB each, so a few hundred fit on Drive comfortably.

In [ ]:
PATCH_DIR = C.INTERIM / 'patches'
records = []
for t in tiles:
    records += patches.cut(t, PATCH_DIR, product_id=t.stem)
print(len(records), 'patches,',
      round(sum(r['bytes'] for r in records) / 1e6, 1), 'MB')
patches.write_index(records, C.REPORTS / 'patch_index.csv')

### Pick the ones worth labelling

Hand labelling is the bottleneck of the whole project, so the patches chosen
must be the informative ones, not the first 300. Mixed patches - those with
both water and land - teach the model about boundaries; a patch of open sea
teaches it almost nothing.

In [ ]:
import pandas as pd
pi = pd.read_csv(C.REPORTS / 'patch_index.csv')

score = []
for name in pi.patch:
    idx = indices.compute(PATCH_DIR / name, which=('mndwi', 'ndvi', 'bsi'))
    w = float((idx['mndwi'] > 0).mean())
    score.append({'patch': name, 'water_frac': round(w, 3),
                  'mixedness': round(1 - abs(2 * w - 1), 3),
                  'bsi_median': round(float(idx['bsi'].mean()), 3)})
sc = pi.merge(pd.DataFrame(score), on='patch')
chosen = sc.sort_values('mixedness', ascending=False).head(300)
chosen.to_csv(DRIVE / 'patch_index_chosen.csv', index=False)
chosen.head(10)

### Copy them to Drive

In [ ]:
import shutil
dest = DRIVE / 'patches'
dest.mkdir(parents=True, exist_ok=True)
n = 0
for name in chosen.patch:
    src_p = PATCH_DIR / name
    if not (dest / name).exists():
        shutil.copy(src_p, dest / name); n += 1
print(n, 'patches copied;',
      round(sum(f.stat().st_size for f in dest.glob('*.tif')) / 1e6, 1), 'MB on Drive')

## What to do next

1. Sync `MyDrive/aral/patches/` to the laptop - tens of megabytes, not gigabytes.
2. Open them in QGIS and draw the four classes. This is the part no amount of
   compute replaces, and it is on the critical path.
3. Put the labels back on Drive. They are the only thing here that cannot be
   regenerated.

Everything else in this notebook is disposable: delete the runtime, and the
manifest rebuilds it.